In [0]:
# # Spark optimizations for large MERGE operations
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.databricks.photon.sort.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.databricks.adaptive.autoOptimizeShuffle.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

In [0]:
from datetime import date
from dateutil.relativedelta import relativedelta

start = date(2024, 12, 1)
end = date(2026, 5, 1)

months = []
current = start
while current <= end:
    months.append(current)
    current += relativedelta(months=1)

months = [i.strftime('%Y-%m-%d') for i in months]

In [0]:
from datetime import date, timedelta

# Day-by-day processing: each day has ~1.5B rows which is manageable
# Dec 2024 is already fixed, start from Jan 1, 2025
start_date = date(2025, 1, 1)
end_date = date(2026, 5, 1)

current = start_date
while current < end_date:
    day_str = current.strftime('%Y-%m-%d')
    result = spark.sql(f"""
        UPDATE prod.datalake_share.enhanced_kinetiq_springserve_commercial
        SET tvid = split_part(tvid, '_', 1)
        WHERE contains(tvid, '_')
          AND date_partition = '{day_str}'
    """)
    # Collect to ensure the UPDATE commits before moving on
    metrics = result.collect()
    print(f"Finished {day_str} | {metrics}")
    current += timedelta(days=1)